# M22 lab solution: The Agent Loop & Tool Calling

**Brief.** Retriever as a tool, by hand: wrap your Part V retriever as a tool, run it through your own tool loop with no framework, and log every tool call and error.

Run with the default environment: `uv run jupyter lab`. The tool loop calls the chat-completions API directly with the `openai` client; LangChain only builds the retriever inside the tool.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

## 1. The retriever as a tool

A tool reaches the model only as a schema: a name, a description and typed parameters. `search_handbook` validates its own arguments and raises on bad ones; the loop, not the tool, decides what the model sees when that happens.

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_handbook",
        "description": "Search the Larkspur Cloud customer handbook. Returns the most relevant passages with their source file.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What to look for, in the customer's words or the handbook's."},
                "k": {"type": "integer", "description": "How many passages to return, 1 to 8.", "default": 4},
            },
            "required": ["query"],
        },
    },
}]


def search_handbook(query: str, k: int = 4) -> str:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    if not isinstance(k, int) or not 1 <= k <= 8:
        raise ValueError(f"k must be an integer from 1 to 8, got {k!r}")
    docs = vectorstore.similarity_search(query, k=k)
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


REGISTRY = {"search_handbook": search_handbook}

## 2. The loop, by hand

While the reply holds tool calls, run each one and send its result back as a `tool` message with the matching `tool_call_id`; a reply without tool calls is the answer. Arguments that are not JSON, unknown tools and exceptions inside a tool all go back to the model as an error message, so it can retry. Rounds are capped. Every call is logged.

`inject_bad_args` replaces the arguments of the run's first tool call with `{"query": "", "k": 99}`, so one run shows a bad argument reaching the tool and being handled.

In [ ]:
import os

from openai import OpenAI

client = OpenAI(base_url="https://api.experientiallabs.ai/v1", api_key=os.environ["EXPERIENTIALLABS_API_KEY"])
MODEL = LLM_NAME  # the same model get_experientiallabs_llm() uses
SYSTEM = ("You answer Larkspur Cloud customer questions. Use search_handbook to find the relevant passages and answer "
          "only from them, citing the source file. If the handbook does not cover the question, say I don't know.")
log = []


def run_tool(call, run_id, round_no, inject_bad_args=False):
    entry = {"run": run_id, "round": round_no, "tool": call.function.name, "args": call.function.arguments}
    t0 = time.perf_counter()
    try:
        args = json.loads(call.function.arguments or "{}")
        if inject_bad_args:
            args = {"query": "", "k": 99}
            entry["args"] = json.dumps(args) + " (injected)"
        fn = REGISTRY.get(call.function.name)
        if fn is None:
            raise KeyError(f"unknown tool {call.function.name!r}")
        content, entry["ok"] = fn(**args), True
    except Exception as err:  # every failure goes back to the model as text
        content, entry["ok"], entry["error"] = f"Error: {type(err).__name__}: {err}", False, str(err)
    entry["ms"] = round((time.perf_counter() - t0) * 1000, 1)
    log.append(entry)
    return {"role": "tool", "tool_call_id": call.id, "content": content}


def ask(question, run_id, max_rounds=6, inject_bad_args=False):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    first = True
    for round_no in range(1, max_rounds + 1):
        reply = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS).choices[0].message
        messages.append(reply.model_dump(exclude_none=True))
        if not reply.tool_calls:
            return reply.content, round_no
        for call in reply.tool_calls:
            messages.append(run_tool(call, run_id, round_no, inject_bad_args and first))
            first = False
    return "Stopped: round limit reached.", max_rounds

## 3. Runs: five golden questions and one forced error

In [ ]:
runs = []
for qid in ["g01", "g02", "g10", "g17", "g26"]:
    reply, rounds = ask(by_id[qid]["question"], qid)
    runs.append({"run": qid, "rounds": rounds, "answer": reply})
    print(qid, f"({rounds} rounds)", reply[:160].replace("\n", " "), "\n")

In [ ]:
reply, rounds = ask(by_id["g05"]["question"], "g05 forced error", inject_bad_args=True)
runs.append({"run": "g05 forced error", "rounds": rounds, "answer": reply})
print(f"({rounds} rounds)", reply)

## Results: the call log

In [ ]:
import pandas as pd

print("run:", time.strftime("%Y-%m-%d"), "| model:", MODEL, "| embeddings:", EMBEDDING_NAME)
print(pd.DataFrame(log).to_string(index=False))
print()
print(pd.DataFrame(runs)[["run", "rounds"]].to_string(index=False))

## What this shows

- The log is the agent's whole behaviour: which tool, with which arguments, in which round, how long it took and whether it failed. Nothing else decides what the model sees.
- In the forced-error run the tool raises, the loop turns the exception into a `tool` message, and the next round shows what the model did with it: a corrected call, or an answer without one.
- The round cap is the loop's only brake. A run that hits it returns a fixed message instead of an answer, which the caller can count.